# GMF candidate-generation baseline

Static implicit-feedback GMF (user/item dot product) trained on session clicks.
The notebook trains both BPR and BCE, evaluates them, and writes Top-500 candidates.


## 1. Setup


In [1]:
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

import numpy as np
import polars as pl
import torch
import torch.nn.functional as F
from torch import nn

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_all.py").is_file())
CG_DIR = ROOT / "5_Models" / "5_01_Candidate_Generation"
sys.path.insert(0, str(CG_DIR))
sys.path.insert(0, str(ROOT / "src"))

from cf_models import load_context, resolve_device
from cg_result_writer import CandidateWriter
from neural_helpers import (
    EarlyStopper,
    EpochNegativeSampler,
    chronological_cutoff,
    chronological_mask,
    max_clicks_per_impression,
    mean_slate_metrics,
    tuning_slates,
)
from newsrec.common import Settings

PROCESSED = ROOT / "1_Data_Preparation" / "processed"
FEATURES = ROOT / "4_Feature_Engineering" / "artifacts" / "prequential-all"
REPRESENTATIONS = ROOT / "3_Representations" / "artifacts"
ARTIFACTS = ROOT / "5_Models" / "5_01_Candidate_Generation" / "artifacts"
RESULTS = ROOT / "results"

TARGET_SPLIT = "validation"
RUN_LOSSES = ("bpr", "bce")
LOSS_CONFIGS = {}
for loss in RUN_LOSSES:
    artifact_dir = ARTIFACTS / f"gmf_{loss}"
    artifact_dir.mkdir(parents=True, exist_ok=True)
    LOSS_CONFIGS[loss] = {
        "model_name": f"gmf_{loss}",
        "run_tag": f"gmf_{loss}",
        "artifact_dir": artifact_dir,
        "recommendations_path": artifact_dir / f"recommendations_{TARGET_SPLIT}_top500.parquet",
        "result_path": RESULTS / f"candidate_generation_gmf_{loss}_{TARGET_SPLIT}.csv",
    }

COMPARISON_PATH = RESULTS / f"candidate_generation_comparison_{TARGET_SPLIT}.csv"

M_VALUES = (50, 100, 200, 500, 750, 1000, 1500)
CANDIDATE_CAP = max(M_VALUES)
EMBED_DIM = 256
NUM_EPOCHS = 50
BATCH_SIZE = 32_768
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-6
SEED = 4222
PATIENCE = 7
TUNE_QUANTILE = 0.9
TUNE_CANDIDATES = PROCESSED / "candidates_train.parquet"
DEVICE = resolve_device("auto")

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Repository:", ROOT)
print("Device:", DEVICE)
print("Torch:", torch.__version__)


Repository: C:\Users\Yong Rui\BT4222-Group-6-
Device: cuda
Torch: 2.13.0a0+rocm7.13.0a20260416


## 2. Data ingestion


Positives are session clicks from the full `labels_train` pool; each training
epoch redraws up to four exposed non-clicks from the same slate.


In [2]:
context = load_context(ROOT)


In [3]:
article_index = pl.read_parquet(
    REPRESENTATIONS / "article_index.parquet",
    columns=["article_row", "news_code"],
).sort("article_row")

run_manifest = json.loads((FEATURES / "run_manifest.json").read_text())
if run_manifest.get("training_candidate_pool") != "complete":
    raise ValueError(
        "Per-epoch sampling needs the complete training pool. "
        "Rebuild stage 4 with the default training_negative_sampling='epoch'."
    )
CFG = Settings(**run_manifest["config"])

TRAINING_ROWS = (
    pl.read_parquet(sorted((FEATURES / "labels_train").glob("*.parquet")))
    .select("impression_uid", "news_code", "clicked")
    .to_pandas()
)
if TRAINING_ROWS["clicked"].isna().any() or not TRAINING_ROWS[
    "clicked"
].isin([0, 1]).all():
    raise ValueError("Training rows must be uniquely binary-labelled")

train_profiles = pl.read_parquet(
    sorted((FEATURES / "request_profiles_train").glob("*.parquet")),
    columns=["impression_uid", "user_code", "timestamp"],
)

positive_uids = np.unique(
    TRAINING_ROWS.loc[TRAINING_ROWS["clicked"].eq(1), "impression_uid"].to_numpy()
)
positive_profiles = train_profiles.filter(
    pl.col("impression_uid").is_in(positive_uids)
)
cutoff_ns = chronological_cutoff(
    positive_profiles["timestamp"].to_numpy(), TUNE_QUANTILE
)
fit_profiles = chronological_mask(train_profiles["timestamp"].to_numpy(), cutoff_ns)
fit_uids = set(
    train_profiles.filter(pl.Series("fit", fit_profiles))["impression_uid"].to_list()
)
FIT_ROWS = TRAINING_ROWS[
    TRAINING_ROWS["impression_uid"].isin(fit_uids)
].reset_index(drop=True)
ALL_ROWS = TRAINING_ROWS
n_groups = int(TRAINING_ROWS["clicked"].sum())
EPOCH_SAMPLER = EpochNegativeSampler(ALL_ROWS, CFG)


def sample_epoch(epoch, *, fit_only):
    """Draw exposed negatives for one epoch and build GMF training arrays."""
    groups, _ = EPOCH_SAMPLER.draw(
        int(epoch), uids=fit_uids if fit_only else None
    )
    groups = (
        pl.from_pandas(groups)
        .join(train_profiles, on="impression_uid", how="left")
        .with_row_index("group_row")
        .join(
            article_index.select(
                pl.col("news_code"),
                pl.col("article_row").alias("pos_row"),
            ),
            left_on="positive_news_code",
            right_on="news_code",
            how="left",
        )
    )
    if groups["user_code"].null_count() or groups["pos_row"].null_count():
        raise ValueError("Sampled groups are missing training profiles or article rows")
    negative_long = (
        groups.select("group_row", "negative_news_codes")
        .explode("negative_news_codes")
        .rename({"negative_news_codes": "negative_news_code"})
        .drop_nulls("negative_news_code")
        .with_columns(pl.int_range(pl.len()).over("group_row").alias("neg_pos"))
        .join(
            article_index.select(
                pl.col("news_code").alias("negative_news_code"),
                pl.col("article_row").alias("neg_row"),
            ),
            on="negative_news_code",
            how="left",
        )
        .sort(["group_row", "neg_pos"])
    )
    if negative_long["neg_row"].null_count():
        raise ValueError("Sampled negatives are missing article rows")
    n_epoch_groups = groups.height
    NEG_MAX = int(groups["negative_news_codes"].list.len().max() or 0)
    if n_epoch_groups and NEG_MAX == 0:
        raise ValueError("Sampled training rows contain no exposed negatives")
    NEG_MATRIX = np.zeros((n_epoch_groups, NEG_MAX), dtype=np.int64)
    group_rows = negative_long["group_row"].to_numpy().astype(np.int64)
    NEG_MATRIX[
        group_rows, negative_long["neg_pos"].to_numpy().astype(np.int64)
    ] = negative_long["neg_row"].to_numpy().astype(np.int64)
    NEG_LEN = np.bincount(group_rows, minlength=n_epoch_groups).astype(np.int64)
    return (
        groups,
        groups["pos_row"].to_numpy().astype(np.int64),
        NEG_MATRIX,
        NEG_LEN,
    )


TUNE_NDCG_K = max_clicks_per_impression(TUNE_CANDIDATES)

tune = tuning_slates(
    TUNE_CANDIDATES,
    cutoff_ns,
    article_index,
    train_profiles.select("impression_uid", "user_code"),
    max_impressions=None,
    seed=SEED + 1,
)

TUNE_UIDS = tune["impression_uid"].to_numpy()
TUNE_USERS = tune["user_code"].to_numpy().astype(np.int64)
TUNE_ITEMS = tune["article_row"].to_numpy().astype(np.int64)
TUNE_CLICKED = tune["clicked"].to_numpy().astype(np.float32)

print(f"Users: {context.n_users:,}")
print(f"Articles: {context.n_items:,}")
print(f"Training rows: {len(TRAINING_ROWS):,}")
print(f"Training positives: {n_groups:,}")
print(f"Chronological fit rows: {len(FIT_ROWS):,} / {len(ALL_ROWS):,}")
print(f"Sampler negatives per positive: up to {CFG.negative_count}")
print(f"Tuning cutoff: {np.datetime64(cutoff_ns, 'ns')}")
print(f"Tuning rows: {tune.height:,}")
print(f"Tuning NDCG k: {TUNE_NDCG_K}")


Users: 50,001
Articles: 51,283
Training rows: 3,389,342
Training positives: 141,558
Chronological fit rows: 3,046,331 / 3,389,342
Sampler negatives per positive: up to 4
Tuning cutoff: 2019-11-12T13:50:10.000000000
Tuning rows: 343,011
Tuning NDCG k: 31


## 3. Model


Two embedding tables and a raw dot product.


In [4]:
class GMF(nn.Module):
    def __init__(self, n_users, n_items, embed_dim):
        super().__init__()
        self.user_embedding = nn.Embedding(n_users, embed_dim)
        self.item_embedding = nn.Embedding(n_items, embed_dim)
        nn.init.normal_(self.user_embedding.weight, std=0.01)
        nn.init.normal_(self.item_embedding.weight, std=0.01)
        with torch.no_grad():
            self.item_embedding.weight[0].zero_()

    def forward(self, user_ids, item_ids):
        return (
            self.user_embedding(user_ids) * self.item_embedding(item_ids)
        ).sum(dim=-1)


## 4. Training


BPR/BCE training, chronological tuning and refit.


In [5]:
def train_epoch(model, optimizer, user_ids, item_ids, neg_matrix, neg_len, rng):
    model.train()
    order = rng.permutation(len(user_ids))
    total = 0.0
    for start in range(0, len(order), BATCH_SIZE):
        batch = order[start : start + BATCH_SIZE]
        users = torch.as_tensor(user_ids[batch], dtype=torch.long, device=DEVICE)
        positives = torch.as_tensor(item_ids[batch], dtype=torch.long, device=DEVICE)
        negatives = torch.as_tensor(neg_matrix[batch], dtype=torch.long, device=DEVICE)
        neg_lens = torch.as_tensor(neg_len[batch], dtype=torch.long, device=DEVICE)
        neg_mask = (
            torch.arange(negatives.shape[1], device=DEVICE)[None, :]
            < neg_lens[:, None]
        )
        optimizer.zero_grad(set_to_none=True)
        positive_scores = model(users, positives)
        negative_scores = model(
            users[:, None].expand(-1, negatives.shape[1]).reshape(-1),
            negatives.reshape(-1),
        ).reshape(len(batch), negatives.shape[1])
        if LOSS == "bpr":
            pair_loss = F.logsigmoid(positive_scores[:, None] - negative_scores)
            loss = -(pair_loss * neg_mask).sum() / neg_mask.sum()
        else:
            loss = F.binary_cross_entropy_with_logits(
                torch.cat([positive_scores, negative_scores[neg_mask]]),
                torch.cat(
                    [
                        torch.ones_like(positive_scores),
                        torch.zeros_like(negative_scores[neg_mask]),
                    ]
                ),
            )
        loss.backward()
        optimizer.step()
        total += float(loss.item()) * len(batch)
    return total / len(order)


In [6]:
def tune_ndcg(model):
    model.eval()
    scores = np.empty(len(TUNE_CLICKED), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, len(TUNE_CLICKED), 200_000):
            end = min(start + 200_000, len(TUNE_CLICKED))
            users = torch.as_tensor(
                TUNE_USERS[start:end], dtype=torch.long, device=DEVICE
            )
            items = torch.as_tensor(
                TUNE_ITEMS[start:end], dtype=torch.long, device=DEVICE
            )
            scores[start:end] = model(users, items).cpu().numpy()
    _, ndcg = mean_slate_metrics(TUNE_UIDS, TUNE_CLICKED, scores, TUNE_NDCG_K)
    return ndcg


In [ ]:
TRAINED_LOSSES = {}
for LOSS in RUN_LOSSES:
    config = LOSS_CONFIGS[LOSS]
    print(f"[{LOSS}] Training {config['model_name']}")
    model = GMF(context.n_users, context.n_items, EMBED_DIM).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    rng = np.random.default_rng(SEED + 1)
    training_log = []
    stopper = EarlyStopper(patience=PATIENCE)

    for epoch in range(1, NUM_EPOCHS + 1):
        started = time.time()
        groups, pos_rows, neg_matrix, neg_len = sample_epoch(
            epoch - 1, fit_only=True
        )
        loss = train_epoch(
            model,
            optimizer,
            groups["user_code"].to_numpy().astype(np.int64),
            pos_rows,
            neg_matrix,
            neg_len,
            rng,
        )
        ndcg = tune_ndcg(model)
        elapsed = time.time() - started
        training_log.append(
            {
                "phase": "tune",
                "epoch": epoch,
                "train_loss": loss,
                "tune_ndcg": ndcg,
                "ndcg_k": TUNE_NDCG_K,
                "seconds": elapsed,
            }
        )
        print(
            f"epoch {epoch:02d} loss={loss:.6f} "
            f"tune_ndcg@{TUNE_NDCG_K}={ndcg:.5f} ({elapsed:.1f}s)"
        )
        if stopper.step(ndcg, model, epoch):
            print("Early stopping")
            break

    stopper.restore(model)
    print(
        f"Best epoch: {stopper.best_epoch} "
        f"(tune_ndcg@{TUNE_NDCG_K}={stopper.best_score:.5f})"
    )

    final_model = GMF(context.n_users, context.n_items, EMBED_DIM).to(DEVICE)
    optimizer = torch.optim.AdamW(
        final_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    rng = np.random.default_rng(SEED + 2)

    for epoch in range(1, stopper.best_epoch + 1):
        started = time.time()
        groups, pos_rows, neg_matrix, neg_len = sample_epoch(
            epoch - 1, fit_only=False
        )
        loss = train_epoch(
            final_model,
            optimizer,
            groups["user_code"].to_numpy().astype(np.int64),
            pos_rows,
            neg_matrix,
            neg_len,
            rng,
        )
        elapsed = time.time() - started
        training_log.append(
            {
                "phase": "refit",
                "epoch": epoch,
                "train_loss": loss,
                "tune_ndcg": float("nan"),
                "ndcg_k": TUNE_NDCG_K,
                "seconds": elapsed,
            }
        )
        print(f"refit epoch {epoch:02d} loss={loss:.6f} ({elapsed:.1f}s)")

    model = final_model
    torch.save(
        {
            "state_dict": model.state_dict(),
            "config": {
                "n_users": context.n_users,
                "n_items": context.n_items,
                "embed_dim": EMBED_DIM,
                "objective": LOSS,
                "negative_sampling": "per-epoch exposed negatives",
                "negative_count": int(CFG.negative_count),
                "negative_max": int(CFG.negative_count),
                "sampler_epoch_base": 0,
                "max_epochs": NUM_EPOCHS,
                "best_epoch": stopper.best_epoch,
                "epochs": stopper.best_epoch,
                "tune_ndcg": stopper.best_score,
                "ndcg_k": TUNE_NDCG_K,
                "batch_size": BATCH_SIZE,
                "learning_rate": LEARNING_RATE,
                "weight_decay": WEIGHT_DECAY,
                "seed": SEED,
                "training_data": "labels_train full training pool",
                "training_groups": n_groups,
            },
        },
        config["artifact_dir"] / "model.pt",
    )
    pl.DataFrame(training_log).write_csv(config["artifact_dir"] / "training_log.csv")
    print("Saved model:", config["artifact_dir"] / "model.pt")
    TRAINED_LOSSES[LOSS] = {
        "model": model,
        "training_log": training_log,
        "config": config,
    }


## 5. Results


Candidate helpers.


In [ ]:
def popularity_select(order, n_eligible, excluded=None, k=CANDIDATE_CAP):
    excluded = set() if excluded is None else set(map(int, excluded))
    selected = [
        int(position)
        for position in order
        if int(position) < n_eligible and int(position) not in excluded
    ]
    return np.asarray(selected[:k], dtype=np.int64)


def popularity_fill(available, popularity, news_code, count):
    pool = np.flatnonzero(available)
    order = np.lexsort((news_code[pool], -popularity[pool]))
    return pool[order[:count]]


def select_top_k(
    scores, popularity, news_code, k, excluded=None, fallback_order=None
):
    scores = np.asarray(scores, dtype=np.float32)
    n = scores.size
    if fallback_order is not None:
        return popularity_select(fallback_order, n, excluded=excluded, k=k)

    available = np.ones(n, dtype=bool)
    if excluded is not None and len(excluded):
        positions = np.asarray(excluded, dtype=np.int64)
        available[positions] = False
        scores = scores.copy()
        scores[positions] = -np.inf

    finite = np.isfinite(scores)
    n_finite = int(np.count_nonzero(finite))
    if n_finite == 0:
        return popularity_fill(available, popularity, news_code, k)

    take = min(k, n_finite)
    if take < n_finite:
        frontier = np.argpartition(-scores, take - 1)[:take]
        threshold = scores[frontier].min()
    else:
        threshold = scores[finite].min()

    above = np.flatnonzero(scores > threshold)
    need = take - above.size
    ties = np.flatnonzero(scores == threshold)
    if need > 0:
        if ties.size > need:
            ties = ties[np.lexsort((news_code[ties], -popularity[ties]))[:need]]
        chosen_positions = np.r_[above, ties]
    else:
        chosen_positions = above

    order = np.lexsort(
        (
            news_code[chosen_positions],
            -popularity[chosen_positions],
            -scores[chosen_positions],
        )
    )
    selected = chosen_positions[order].astype(np.int64)

    if selected.size < k:
        remaining = available.copy()
        remaining[selected] = False
        fill = popularity_fill(remaining, popularity, news_code, k - selected.size)
        selected = np.r_[selected, fill]

    return selected


### Score the candidate universe and write Top-500 candidates

Static user embeddings, eligibility by request timestamp, full history exclusion,
and popularity fallback for empty-history requests.


In [ ]:
def load_profiles(split):
    parts = sorted((FEATURES / f"request_profiles_{split}").glob("*.parquet"))
    return pl.read_parquet(
        parts,
        columns=[
            "impression_uid",
            "user_code",
            "timestamp",
            "full_history_rows",
        ],
    ).sort("timestamp", maintain_order=True)


catalogue = pl.DataFrame(
    {
        "article_row": np.arange(1, context.n_items, dtype=np.int32),
        "news_code": context.news_code[1:],
        "available_from": context.available_from[1:],
        "popularity": context.popularity[1:],
    }
)
ordered = catalogue.sort(["available_from", "news_code"])
article_rows = ordered["article_row"].to_numpy().astype(np.int64)
news_sorted = ordered["news_code"].to_numpy().astype(np.int64)
available_sorted = ordered["available_from"].to_numpy().astype("datetime64[ns]")
popularity_sorted = ordered["popularity"].to_numpy().astype(np.float32)
pos_in_sorted = np.full(context.n_items, -1, dtype=np.int64)
pos_in_sorted[article_rows] = np.arange(article_rows.size, dtype=np.int64)
global_popularity_order = np.lexsort((news_sorted, -popularity_sorted))

profiles = load_profiles(TARGET_SPLIT)
impression_uids = profiles["impression_uid"].to_numpy()
target_users = profiles["user_code"].to_numpy().astype(np.int64)
timestamps = profiles["timestamp"].to_numpy().astype("datetime64[ns]")
history_rows = profiles["full_history_rows"].to_list()
has_history = np.fromiter(
    (len(rows) > 0 for rows in history_rows),
    dtype=bool,
    count=len(history_rows),
)

starts = np.flatnonzero(np.r_[True, timestamps[1:] != timestamps[:-1]])
ends = np.r_[starts[1:], len(timestamps)]
batches = list(zip(starts, ends))
print(f"Scoring requests: {len(timestamps):,}")

cutoff_values = np.arange(1, CANDIDATE_CAP + 1, dtype=np.int16)

for loss in RUN_LOSSES:
    config = LOSS_CONFIGS[loss]
    model = TRAINED_LOSSES[loss]["model"]
    print(f"[{loss}] Scoring {config['model_name']}")
    user_emb = model.user_embedding.weight.detach().cpu().numpy()[target_users]
    item_emb = model.item_embedding.weight.detach().cpu().numpy()[article_rows]
    writer = CandidateWriter(config["recommendations_path"])
    started = time.time()
    try:
        for batch_no, (start, end) in enumerate(batches, 1):
            timestamp = timestamps[start]
            n_eligible = int(
                np.searchsorted(available_sorted, timestamp, side="right")
            )
            scores = user_emb[start:end] @ item_emb[:n_eligible].T
            popularity_batch = popularity_sorted[:n_eligible]
            news_batch = news_sorted[:n_eligible]
            selected_batch = np.empty(
                (end - start, CANDIDATE_CAP), dtype=np.int64
            )
            for local, profile_row in enumerate(range(start, end)):
                rows = history_rows[profile_row]
                if rows:
                    excluded = pos_in_sorted[np.asarray(rows, dtype=np.int64)]
                    excluded = excluded[
                        (excluded >= 0) & (excluded < n_eligible)
                    ]
                else:
                    excluded = np.empty(0, dtype=np.int64)
                fallback = (
                    None
                    if has_history[profile_row]
                    else global_popularity_order
                )
                selected_batch[local] = select_top_k(
                    scores[local],
                    popularity_batch,
                    news_batch,
                    CANDIDATE_CAP,
                    excluded=excluded,
                    fallback_order=fallback,
                )

            writer.add(
                np.repeat(impression_uids[start:end], CANDIDATE_CAP),
                np.tile(cutoff_values, end - start),
                news_batch[selected_batch.reshape(-1)],
            )
            if batch_no % 1000 == 0 or batch_no == len(batches):
                print(
                    f"Scored {batch_no:,}/{len(batches):,} timestamp batches; "
                    f"{writer.rows:,} rows",
                    flush=True,
                )
    finally:
        writer.close()

    elapsed = time.time() - started
    print(f"Wrote {len(timestamps) * CANDIDATE_CAP:,} candidate rows in {elapsed:.1f}s")
    print("Candidates:", config["recommendations_path"])
    TRAINED_LOSSES[loss]["recommendations_path"] = config["recommendations_path"]
    TRAINED_LOSSES[loss]["score_seconds"] = elapsed


### Evaluation and comparison


In [ ]:
from cg_evaluation import CandidateEvaluator

for loss in RUN_LOSSES:
    config = LOSS_CONFIGS[loss]
    evaluator = CandidateEvaluator(root=ROOT, split=TARGET_SPLIT)
    evaluation = evaluator.evaluate_to_csv(
        TRAINED_LOSSES[loss]["recommendations_path"],
        model_name=config["model_name"],
        m_values=M_VALUES,
        output_dir=RESULTS,
    )
    print(f"[{loss}]")
    print(evaluation.sort("m"))
    print("Results:", config["result_path"])


In [ ]:
model_names = [LOSS_CONFIGS[loss]["model_name"] for loss in RUN_LOSSES]
comparison = (
    pl.read_csv(COMPARISON_PATH)
    .filter(pl.col("model").is_in(model_names))
    .filter(
        pl.col("metric").is_in(
            [
                "recall_pooled",
                "hit_rate",
                "catalogue_coverage",
                "personalisation",
                "long_tail_recall",
                "cold_article_recall",
            ]
        )
    )
    .pivot(on="model", index=["metric", "m"], values="value")
    .sort(["metric", "m"])
)
print(comparison)


## Notes

- The user representation is a static `user_code` embedding.
- Positives are session clicks; negatives are observed non-clicks from the same slate.
- Up to four exposed negatives are redrawn from the same slate each epoch.
- The dot product is a ranking score, not a calibrated click probability.
